# 삼성전자 재무제표 분석 + 종목 스크리닝

KoFinance SDK를 사용하여:
1. 삼성전자 3년 재무 트렌드 분석
2. 조건 기반 종목 스크리닝

**필요 패키지**: `pip install kofinance matplotlib`

In [ ]:
from kofinance import KoFinance
import matplotlib.pyplot as plt
import matplotlib
import os

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 클라이언트 초기화
kf = KoFinance(os.environ.get('KOFINANCE_API_KEY', 'your-api-key'))
print('KoFinance SDK 연결 완료')

---
## Part 1: 삼성전자 3년 재무 트렌드

In [ ]:
# 삼성전자 3년치 연간 재무제표
df = kf.financials('005930', period='3y', type='annual')

# 주요 컬럼만 확인
cols = ['period', 'is_revenue', 'is_operating_income', 'is_net_income',
        'ratio_roe', 'ratio_per', 'ratio_debt_ratio', 'ratio_operating_margin']
df[cols]

In [ ]:
# 매출액 & 영업이익 추이
fig, ax1 = plt.subplots(figsize=(10, 6))

x = df['period']
revenue = df['is_revenue'] / 1e12
op_income = df['is_operating_income'] / 1e12

# 매출액 (막대)
bars = ax1.bar(x, revenue, alpha=0.7, color='#4A90D9', label='매출액', width=0.5)
ax1.set_ylabel('매출액 (조원)', color='#4A90D9', fontsize=12)

# 막대 위에 값 표시
for bar, val in zip(bars, revenue):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val:.0f}조', ha='center', fontsize=10, color='#4A90D9')

# 영업이익 (선)
ax2 = ax1.twinx()
ax2.plot(x, op_income, 'r-o', linewidth=2.5, markersize=10, label='영업이익')
ax2.set_ylabel('영업이익 (조원)', color='red', fontsize=12)

for xi, yi in zip(x, op_income):
    ax2.annotate(f'{yi:.0f}조', (xi, yi), textcoords='offset points',
                 xytext=(0, 12), ha='center', color='red', fontsize=10)

# 범례
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

plt.title('삼성전자 매출액 및 영업이익 추이', fontsize=14, fontweight='bold')
ax1.set_xlabel('회계연도', fontsize=12)
ax1.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 수익성 지표 대시보드
latest = df.iloc[0]

print('=' * 50)
print(f'  삼성전자 {latest["period"]}년 주요 지표')
print('=' * 50)
print(f'  매출액:       {latest["is_revenue"]/1e12:>10.1f} 조원')
print(f'  영업이익:     {latest["is_operating_income"]/1e12:>10.1f} 조원')
print(f'  순이익:       {latest["is_net_income"]/1e12:>10.1f} 조원')
print('-' * 50)
print(f'  ROE:          {latest["ratio_roe"]:>10.1f} %')
print(f'  PER:          {latest["ratio_per"]:>10.1f} 배')
print(f'  영업이익률:   {latest["ratio_operating_margin"]:>10.1f} %')
print(f'  부채비율:     {latest["ratio_debt_ratio"]:>10.1f} %')
print('=' * 50)

In [ ]:
# ROE & 부채비율 추이
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROE
axes[0].plot(df['period'], df['ratio_roe'], 'g-o', linewidth=2, markersize=8)
axes[0].fill_between(df['period'], df['ratio_roe'], alpha=0.15, color='green')
axes[0].axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='10% 기준')
for xi, yi in zip(df['period'], df['ratio_roe']):
    axes[0].annotate(f'{yi:.1f}%', (xi, yi), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=10)
axes[0].set_title('ROE 추이', fontsize=13, fontweight='bold')
axes[0].set_ylabel('ROE (%)')
axes[0].set_xlabel('회계연도')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 부채비율
colors = ['#FF6B6B' if v > 100 else '#4ECDC4' for v in df['ratio_debt_ratio']]
axes[1].bar(df['period'], df['ratio_debt_ratio'], color=colors, alpha=0.8)
axes[1].axhline(y=100, color='red', linestyle='--', alpha=0.5, label='100% 기준')
for i, (xi, yi) in enumerate(zip(df['period'], df['ratio_debt_ratio'])):
    axes[1].text(xi, yi + 1, f'{yi:.0f}%', ha='center', fontsize=10)
axes[1].set_title('부채비율 추이', fontsize=13, fontweight='bold')
axes[1].set_ylabel('부채비율 (%)')
axes[1].set_xlabel('회계연도')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 2: 종목 스크리닝

조건 기반으로 투자 후보 종목을 찾아봅니다.

In [ ]:
# 가치주 스크리닝: ROE 높고, PER 낮고, 배당 있는 종목
value_stocks = kf.screen(
    roe_gt=12,
    per_lt=15,
    dividend_yield_gt=2.0,
    market='KOSPI',
)

print(f'조건에 맞는 종목: {len(value_stocks)}개\n')
value_stocks[['symbol', 'name', 'roe', 'per', 'pbr', 'dividend_yield', 'market_cap']].head(10)

In [ ]:
# 스크리닝 결과 시각화: ROE vs PER 산점도
if not value_stocks.empty:
    fig, ax = plt.subplots(figsize=(10, 7))

    scatter = ax.scatter(
        value_stocks['per'],
        value_stocks['roe'],
        s=value_stocks['market_cap'] / value_stocks['market_cap'].max() * 500 + 30,
        alpha=0.6,
        c=value_stocks['dividend_yield'],
        cmap='YlOrRd',
        edgecolors='gray',
        linewidth=0.5,
    )

    # 상위 5개 종목 이름 표시
    top5 = value_stocks.nlargest(5, 'roe')
    for _, row in top5.iterrows():
        ax.annotate(
            row['name'],
            (row['per'], row['roe']),
            textcoords='offset points',
            xytext=(8, 5),
            fontsize=9,
        )

    plt.colorbar(scatter, label='배당수익률 (%)')
    ax.set_xlabel('PER (배)', fontsize=12)
    ax.set_ylabel('ROE (%)', fontsize=12)
    ax.set_title('KOSPI 가치주 스크리닝 (ROE > 12%, PER < 15)', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('조건에 맞는 종목이 없습니다.')

In [ ]:
# 저PBR 스크리닝: 자산 대비 저평가 종목
low_pbr = kf.screen(
    pbr_lt=0.7,
    roe_gt=8,
    debt_ratio_lt=100,
    market='KOSPI',
)

print(f'저PBR 가치주: {len(low_pbr)}개\n')
low_pbr[['symbol', 'name', 'pbr', 'roe', 'per', 'debt_ratio']].head(10)

---
## Part 3: 동종업계 비교

삼성전자와 SK하이닉스의 재무 성과를 비교합니다.

In [ ]:
# 삼성전자 + SK하이닉스 동시 조회
compare = kf.financials(['005930', '000660'], period='3y')

compare[['symbol', 'name', 'period', 'is_revenue', 'is_operating_income', 'ratio_roe']]

In [ ]:
# 매출 & ROE 비교 차트
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for symbol, group in compare.groupby('symbol'):
    name = group.iloc[0]['name']

    # 매출 비교
    axes[0].plot(group['period'], group['is_revenue'] / 1e12, '-o',
                 label=name, linewidth=2, markersize=7)

    # ROE 비교
    axes[1].plot(group['period'], group['ratio_roe'], '-o',
                 label=name, linewidth=2, markersize=7)

axes[0].set_title('매출액 비교 (조원)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('매출액 (조원)')
axes[0].set_xlabel('회계연도')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('ROE 비교 (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('ROE (%)')
axes[1].set_xlabel('회계연도')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 결론

이 노트북에서 확인한 것:

1. **재무 트렌드 분석**: `kf.financials()` 한 줄로 3년치 재무제표를 DataFrame으로 받아 시각화
2. **종목 스크리닝**: `kf.screen()` 한 줄로 ROE/PER/배당 조건에 맞는 종목 필터링
3. **동종업계 비교**: 종목 리스트를 넘기면 여러 기업을 한번에 비교

pykrx에서 수십 줄 + 수십 분이 걸리던 작업이, KoFinance에서는 1~2줄 + 1초로 끝납니다.

- API 키 발급: [kofinance.ntriq.co.kr](https://kofinance.ntriq.co.kr)
- 전체 문서: [kofinance.ntriq.co.kr/docs](https://kofinance.ntriq.co.kr/docs)